# Hi-EF Phase 1: label-oracle statistical audit

CPU-only validation audit of the context, raw-clip-III, and Party-A emotion-label oracle models. It also audits both label-permutation controls. The analysis uses paired predictions and a fixed 5,000-replicate hierarchical bootstrap; it never evaluates the test partition.

Attach the saved outputs of `ptrnghieu/hief-multiseed-validation` and the label-oracle notebook. Internet is required only to clone the repository.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
OUTPUT = Path('/kaggle/working/phase1_label_oracle_statistics')

if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'], check=True)
else:
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# Locate both saved notebook outputs by their signed summaries and prediction files.
import json

baseline_roots = []
for path in Path('/kaggle/input').rglob('validation_matrix_summary.json'):
    try:
        summary = json.loads(path.read_text())
    except Exception:
        continue
    root = path.parent
    if (
        summary.get('test_evaluated') is False
        and summary.get('seeds') == [42, 123, 456, 789, 1024]
        and all((root / f'context_seed{seed}' / 'val_predictions.npz').exists() for seed in summary['seeds'])
        and all((root / f'full_seed{seed}' / 'val_predictions.npz').exists() for seed in summary['seeds'])
    ):
        baseline_roots.append(root)

oracle_roots = []
for path in Path('/kaggle/input').rglob('label_oracle_summary.json'):
    try:
        summary = json.loads(path.read_text())
    except Exception:
        continue
    root = path.parent
    if (
        summary.get('test_evaluated') is False
        and summary.get('model_seeds') == [42, 123, 456, 789, 1024]
        and all((root / 'runs' / f'oracle_label_seed{seed}' / 'val_predictions.npz').exists() for seed in summary['model_seeds'])
    ):
        oracle_roots.append(root)

if len(baseline_roots) != 1 or len(oracle_roots) != 1:
    raise RuntimeError(
        'Attach exactly one saved baseline output and one saved label-oracle output; '
        f'found baseline={baseline_roots}, oracle={oracle_roots}'
    )
BASELINE = baseline_roots[0]
ORACLE = oracle_roots[0]
print('Baseline:', BASELINE)
print('Oracle:  ', ORACLE)

In [ ]:
command = [
    'python', str(REPO / 'experiments/analyze_label_oracle.py'),
    '--baseline-dir', str(BASELINE),
    '--label-oracle-dir', str(ORACLE),
    '--manifest', str(REPO / 'experiments/manifests/source_folder_split_seed42.csv'),
    '--output-dir', str(OUTPUT),
    '--bootstrap-replicates', '5000',
    '--bootstrap-seed', '3901',
]
subprocess.run(command, check=True)

In [ ]:
import pandas as pd

summary = json.loads((OUTPUT / 'oracle_statistical_summary.json').read_text())
assert summary['test_evaluated'] is False
display(pd.read_csv(OUTPUT / 'model_comparison_effects_by_seed.csv'))
display(pd.read_csv(OUTPUT / 'label_control_effects_by_seed.csv'))
print(json.dumps({
    'model_comparisons': summary['model_comparisons'],
    'label_controls': summary['label_controls'],
    'hierarchical_bootstrap': summary['hierarchical_bootstrap'],
}, indent=2))

Run via **Save Version → Save & Run All** with a CPU accelerator. Preserve the complete `/kaggle/working/phase1_label_oracle_statistics` directory. Bootstrap intervals are validation uncertainty diagnostics, not confirmatory test-set estimates.